In [6]:
import pandas as pd
import numpy as np

DATASETS = ["cars3d", "dsprites", "mpi3d", "clevr", "iraven", "shapes3d"]

# Diccionarios para despliegue (ajústalos a tu gusto)
MODEL_NAME_MAP = {
    "resnet18": "ResNet-18",
    "resnet18_mixer": "LICG(ResNet-18)",
    "resnet18_mixer_rp64_all_cases": "LICG(ResNet-18)-64",
    "resnet18_mixer_rp128_all_cases": "LICG(ResNet-18)-128",
    "resnet18_mixer_rp256_all_cases": "LICG(ResNet-18)-256",
    "resnet18_mixer_rp512_all_cases": "LICG(ResNet-18)-512",
    "split": "AIN",
    "split_4x": "AIN (4x)",
    "split_resnet_mixer": "LCIG(AIN)",
    "split_resnet_mixer_red_64": "LCIG(AIN)-64",
    "split_resnet_mixer_red_128": "LCIG(AIN)-128",
    "split_resnet_mixer_red_256": "LCIG(AIN)-256",
    "split_resnet_mixer_iid": "LCIG(AIN)+IID",
    "split_resnet_mixer_no_mixer": "LCIG(AIN) - MIXER",
    "split_resnet_mixer_no_mixer_iid": "LCIG(AIN) - MIXER + IID",
    "split_resnet_mixer_all_cases": "LCIG(AIN) [ALL]",
    "split_resnet_mixer_all_cases_iid": "LCIG(AIN) [ALL] + IID",
    "ed": "ED",
    "ed_mixer_rp64": "LCIG(ED)-64",
    "ed_mixer_rp128": "LCIG(ED)-128",
    "ed_mixer_rp256": "LCIG(ED)-256",
}


# Modelos a ignorar
IGNORE_ARCHS = {
    "split_1",
    "split_2",
    "split_3",
    "split_4",
    *{
        f"split_resnet_mixer_s{s}_rp{rp}_all_cases"
        for s in [1, 2, 3, 4]
        for rp in [64, 128, 256]
    },
}

DATASET_NAME_MAP = {
    "cars3d": "C3D",
    "dsprites": "dSprites",
    "mpi3d": "MPI3D",
    "clevr": "CLEVR",
    "iraven": "I-RAVEN",
    "shapes3d": "Sh3D",
}

# Columnas de métricas (para excluirlas al definir "config")
METRIC_COLS = ["train_acc", "val_acc", "ood_val_0_acc", "test_acc"]

DECIMALS = 2
BOLD_TOL = 1e-12  # tolerancia para empates/floating
def fmt(mean, std, decimals=DECIMALS):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{decimals}f} (—)"
    return f"{mean:.{decimals}f} ({std:.{decimals}f})"


## Cálculo Tabla Resultados (ignorando ablation de MPI3D)

In [19]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
DECIMALS = 2
BOLD_TOL = 1e-12   # tolerancia para comparar empates
STAR_TOL = 1e-12   # tolerancia para el *

# Si quieres cambiar etiquetas visibles, hazlo acá
DISPLAY_NAME_MAP = MODEL_NAME_MAP.copy()

# Filas sintéticas agregadas por familia
AGG_FAMILIES = {
    "__lcig_resnet18__": {
        "label": MODEL_NAME_MAP.get("resnet18_mixer", "LCIG(ResNet-18)"),
        "members": [
            #"resnet18_mixer",
            "resnet18_mixer_rp64_all_cases",
            "resnet18_mixer_rp128_all_cases",
            "resnet18_mixer_rp256_all_cases",
            "resnet18_mixer_rp512_all_cases",
        ],
    },
    "__lcig_ain__": {
        "label": MODEL_NAME_MAP.get("split_resnet_mixer", "LCIG(AIN)"),
        "members": [
            "split_resnet_mixer_red_64",
            "split_resnet_mixer_red_128",
            "split_resnet_mixer_red_256",
        ],
    },
    "__lcig_ed__": {
        "label": "LCIG(ED)",
        "members": [
            "ed_mixer_rp64",
            "ed_mixer_rp128",
            "ed_mixer_rp256",
        ],
    },
}

DISPLAY_NAME_MAP.update({
    "__lcig_resnet18__": AGG_FAMILIES["__lcig_resnet18__"]["label"],
    "__lcig_ain__": AGG_FAMILIES["__lcig_ain__"]["label"],
    "__lcig_ed__": AGG_FAMILIES["__lcig_ed__"]["label"],
})

# Orden de filas visibles en la tabla final
ROW_GROUPS = [
    ["resnet18", "__lcig_resnet18__"],
    ["split", "__lcig_ain__"],
    ["ed", "__lcig_ed__"],
]

# ============================================================
# HELPERS
# ============================================================
def safe_score(mean, std):
    """Criterio de selección: mean - std; si std es NaN, penalización 0."""
    if pd.isna(mean):
        return np.nan
    return mean - (0.0 if pd.isna(std) else std)

def build_cell(mean, std, decimals=2):
    return fmt(mean, std, decimals=decimals)

def build_md_cell(text, bold=False, star=False):
    if text == "":
        return text
    if star:
        text = text + "*"
    if bold:
        text = f"**{text}**"
    return text

def build_tex_cell(text, bold=False, star=False):
    if text == "":
        return text
    if star:
        text = text + r"$^\ast$"
    if bold:
        text = r"\textbf{" + text + "}"
    return text
def plot_table_results(method):

    # ============================================================
    # 1) Mejor config por arch y dataset usando score = mean - std
    # ============================================================
    records = []
    
    for dataset in DATASETS:
        df = pd.read_pickle(f"{dataset}_{method}.pkl").copy()
    
        if "arch" not in df.columns or "seed" not in df.columns or "test_acc" not in df.columns:
            raise ValueError(f"{dataset}_id.pkl debe contener columnas: arch, seed, test_acc")
    
        # Todas las columnas no métricas definen la configuración, excepto seed
        non_metric_cols = [c for c in df.columns if c not in METRIC_COLS]
        config_cols = [c for c in non_metric_cols if c != "seed"]
    
        # Promedio por seed dentro de cada configuración
        per_seed = (
            df.groupby(config_cols + ["seed"], dropna=False)["test_acc"]
            .mean()
            .reset_index()
        )
    
        # Estadísticos entre seeds para cada configuración
        stats = (
            per_seed.groupby(config_cols, dropna=False)["test_acc"]
            .agg(mean="mean", std="std", n="count")
            .reset_index()
        )
    
        stats["score"] = stats.apply(lambda r: safe_score(r["mean"], r["std"]), axis=1)
    
        # Mejor configuración por arch según mean - std
        best_idx = stats.groupby("arch")["score"].idxmax()
        best = stats.loc[best_idx].copy()
    
        for _, r in best.iterrows():
            records.append({
                "arch": r["arch"],
                "dataset": dataset,
                "mean": r["mean"],
                "std": r["std"],
                "score": r["score"],
            })
    
    long_df = pd.DataFrame(records)
    
    # ============================================================
    # 2) Construir filas sintéticas LCIG(X) a partir de variantes -Y
    # ============================================================
    agg_records = []
    
    for agg_arch, spec in AGG_FAMILIES.items():
        fam_df = long_df[long_df["arch"].isin(spec["members"])].copy()
        if fam_df.empty:
            continue
    
        for dataset in DATASETS:
            sub = fam_df[fam_df["dataset"] == dataset].copy()
            if sub.empty:
                continue
    
            best_row = sub.loc[sub["score"].idxmax()]
            agg_records.append({
                "arch": agg_arch,
                "dataset": dataset,
                "mean": best_row["mean"],
                "std": best_row["std"],
                "score": best_row["score"],
                "source_arch": best_row["arch"],  # útil por si luego quieres saber qué Y ganó
            })
    
    agg_df = pd.DataFrame(agg_records)
    
    # Todas las filas "reales" + agregadas
    all_df = pd.concat([long_df, agg_df], ignore_index=True, sort=False)
    
    # ============================================================
    # 3) Nos quedamos con la tabla resumen:
    #    baseline X + agregado LCIG(X)
    # ============================================================
    summary_arches = [a for group in ROW_GROUPS for a in group]
    summary_df = all_df[all_df["arch"].isin(summary_arches)].copy()
    
    mean_df = summary_df.pivot(index="arch", columns="dataset", values="mean").reindex(
        index=summary_arches, columns=DATASETS
    )
    std_df = summary_df.pivot(index="arch", columns="dataset", values="std").reindex(
        index=summary_arches, columns=DATASETS
    )
    score_df = summary_df.pivot(index="arch", columns="dataset", values="score").reindex(
        index=summary_arches, columns=DATASETS
    )
    
    # ============================================================
    # 4) Bold:
    #    máximo SOLO entre baseline X y LCIG(X), por dataset
    # ============================================================
    best_summary_score = score_df.max(axis=0, skipna=True)
    is_bold = score_df.sub(best_summary_score, axis=1).abs() <= BOLD_TOL
    
    # ============================================================
    # 5) Star:
    #    máximo absoluto entre TODOS los modelos disponibles, por dataset
    # ============================================================
    all_score_df = all_df.pivot(index="arch", columns="dataset", values="score").reindex(columns=DATASETS)
    best_global_score = all_score_df.max(axis=0, skipna=True)
    is_star = score_df.sub(best_global_score, axis=1).abs() <= STAR_TOL
    
    # ============================================================
    # 6) Tabla base (strings)
    # ============================================================
    plain = pd.DataFrame(index=summary_arches, columns=DATASETS, dtype=object)
    
    for ds in DATASETS:
        for arch in summary_arches:
            plain.loc[arch, ds] = build_cell(
                mean_df.loc[arch, ds],
                std_df.loc[arch, ds],
                decimals=DECIMALS,
            )
    
    # ============================================================
    # 7) NOTEBOOK DISPLAY
    # ============================================================
    plain_nb = plain.copy()
    
    for ds in DATASETS:
        for arch in summary_arches:
            txt = plain_nb.loc[arch, ds]
            if txt != "" and bool(is_star.loc[arch, ds]):
                plain_nb.loc[arch, ds] = txt + "*"
    
    plain_nb = plain_nb.rename(index=DISPLAY_NAME_MAP, columns=DATASET_NAME_MAP)
    
    is_bold_nb = (
        is_bold.rename(index=DISPLAY_NAME_MAP, columns=DATASET_NAME_MAP)
        .reindex(index=plain_nb.index, columns=plain_nb.columns)
        .fillna(False)
    )
    
    display(
        plain_nb.style.apply(
            lambda col: [
                "font-weight: bold" if bool(is_bold_nb.loc[idx, col.name]) else ""
                for idx in col.index
            ],
            axis=0,
        )
    )
    
    # ============================================================
    # 8) MARKDOWN
    # ============================================================
    md = plain.copy()
    
    for ds in DATASETS:
        for arch in summary_arches:
            md.loc[arch, ds] = build_md_cell(
                md.loc[arch, ds],
                bold=bool(is_bold.loc[arch, ds]),
                star=bool(is_star.loc[arch, ds]),
            )
    
    md = md.rename(index=DISPLAY_NAME_MAP, columns=DATASET_NAME_MAP)
    print(md.to_markdown())
    
    # ============================================================
    # 9) LATEX con midrules entre grupos
    # ============================================================
    tex = plain.copy()
    
    for ds in DATASETS:
        for arch in summary_arches:
            tex.loc[arch, ds] = build_tex_cell(
                tex.loc[arch, ds],
                bold=bool(is_bold.loc[arch, ds]),
                star=bool(is_star.loc[arch, ds]),
            )
    
    tex = tex.rename(index=DISPLAY_NAME_MAP, columns=DATASET_NAME_MAP)
    
    dataset_headers = [DATASET_NAME_MAP.get(ds, ds) for ds in DATASETS]
    colspec = "l" + "c" * len(DATASETS)
    
    latex_lines = []
    latex_lines.append(r"\begin{table}[t]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\small")
    latex_lines.append(r"\setlength{\tabcolsep}{4pt}")
    latex_lines.append(r"\renewcommand{\arraystretch}{1.1}")
    latex_lines.append(r"\resizebox{\textwidth}{!}{%")
    latex_lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    latex_lines.append(r"\toprule")
    latex_lines.append("Dataset & " + " & ".join(dataset_headers) + r" \\")
    latex_lines.append(r"\midrule")
    
    for g_idx, group in enumerate(ROW_GROUPS):
        visible_rows = [arch for arch in group if arch in plain.index]
        if not visible_rows:
            continue
    
        for arch in visible_rows:
            row_name = DISPLAY_NAME_MAP.get(arch, arch)
            cells = [tex.loc[row_name, DATASET_NAME_MAP.get(ds, ds)] for ds in DATASETS]
            latex_lines.append(row_name + " & " + " & ".join(cells) + r" \\")
    
        if g_idx < len(ROW_GROUPS) - 1:
            latex_lines.append(r"\midrule")
    
    latex_lines.append(r"\bottomrule")
    latex_lines.append(r"\end{tabular}")
    latex_lines.append(r"}")
    latex_lines.append(r"\caption{TODO: caption}")
    latex_lines.append(r"\label{tab:oracle_results}")
    latex_lines.append(r"\end{table}")
    
    latex_small = "\n".join(latex_lines)
    print(latex_small)

In [20]:
plot_table_results("id")

,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
R18,32.12 (1.55),20.76 (0.93),42.82 (0.88),27.14 (6.72),11.26 (3.49),86.44 (1.05)
LICG(R18),41.37 (1.39),24.32 (2.38),45.21 (0.54),36.97 (3.67),16.69 (2.63),86.14 (1.53)
AIN,43.56 (0.96),61.98 (1.80),55.73 (0.31),53.77 (2.60),63.63 (1.96),85.02 (1.61)
LCIG(AIN),51.05 (2.18)*,65.83 (0.22)*,55.57 (0.49),63.06 (4.94),70.74 (3.69),93.61 (4.72)
ED,45.44 (1.56),63.12 (1.79),59.51 (5.82),53.39 (3.04),74.89 (3.64),98.15 (1.20)*
LCIG(ED),49.94 (1.28),53.68 (5.02),60.65 (3.71),63.33 (2.13)*,73.37 (0.94)*,96.66 (1.48)


|           | C3D               | dSprites          | MPI3D            | CLEVR             | I-RAVEN           | Sh3D              |
|:----------|:------------------|:------------------|:-----------------|:------------------|:------------------|:------------------|
| R18       | 32.12 (1.55)      | 20.76 (0.93)      | 42.82 (0.88)     | 27.14 (6.72)      | 11.26 (3.49)      | 86.44 (1.05)      |
| LICG(R18) | 41.37 (1.39)      | 24.32 (2.38)      | 45.21 (0.54)     | 36.97 (3.67)      | 16.69 (2.63)      | 86.14 (1.53)      |
| AIN       | 43.56 (0.96)      | 61.98 (1.80)      | 55.73 (0.31)     | 53.77 (2.60)      | 63.63 (1.96)      | 85.02 (1.61)      |
| LCIG(AIN) | **51.05 (2.18)*** | **65.83 (0.22)*** | 55.57 (0.49)     | 63.06 (4.94)      | 70.74 (3.69)      | 93.61 (4.72)      |
| ED        | 45.44 (1.56)      | 63.12 (1.79)      | 59.51 (5.82)     | 53.39 (3.04)      | 74.89 (3.64)      | **98.15 (1.20)*** |
| LCIG(ED)  | 49.94 (1.28)      | 53.68 (5.02)      | **60.65 (3.71)*

In [14]:
plot_table_results("oracle")

,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
R18,35.59,22.71 (0.76),43.59 (0.22),30.70 (4.54),17.98 (7.03),86.44 (1.05)
LICG(R18),44.68 (1.50),25.04 (2.02),47.15 (0.54),44.14 (4.44),23.34 (2.80),92.11 (1.66)
AIN,46.27 (1.33),62.06 (1.66),55.80 (0.34),61.76 (2.38),76.96 (3.25),87.56 (2.28)
LCIG(AIN),56.85 (1.76)*,67.60 (2.21)*,57.30 (2.30),68.80 (2.24)*,78.94 (2.13),95.86 (1.82)
ED,50.36 (1.29),64.29 (1.32),62.37 (7.98),64.49 (4.05),84.23 (3.20),98.85 (0.52)*
LCIG(ED),53.92 (0.80),56.66 (2.78),65.71 (2.83),69.58 (3.41),88.72 (1.27)*,98.28 (0.44)


|           | C3D               | dSprites          | MPI3D            | CLEVR             | I-RAVEN           | Sh3D              |
|:----------|:------------------|:------------------|:-----------------|:------------------|:------------------|:------------------|
| R18       | 35.59             | 22.71 (0.76)      | 43.59 (0.22)     | 30.70 (4.54)      | 17.98 (7.03)      | 86.44 (1.05)      |
| LICG(R18) | 44.68 (1.50)      | 25.04 (2.02)      | 47.15 (0.54)     | 44.14 (4.44)      | 23.34 (2.80)      | 92.11 (1.66)      |
| AIN       | 46.27 (1.33)      | 62.06 (1.66)      | 55.80 (0.34)     | 61.76 (2.38)      | 76.96 (3.25)      | 87.56 (2.28)      |
| LCIG(AIN) | **56.85 (1.76)*** | **67.60 (2.21)*** | 57.30 (2.30)     | **68.80 (2.24)*** | 78.94 (2.13)      | 95.86 (1.82)      |
| ED        | 50.36 (1.29)      | 64.29 (1.32)      | 62.37 (7.98)     | 64.49 (4.05)      | 84.23 (3.20)      | **98.85 (0.52)*** |
| LCIG(ED)  | 53.92 (0.80)      | 56.66 (2.78)      | **65.71 (2.83)*

## Ablación MPI3d sobre cantidad de capas a considerar en AIN

In [15]:
import re
import pandas as pd
import numpy as np

# =========================
# CONFIG
# =========================
DATASET = "mpi3d"
DECIMALS = 2
BOLD_TOL = 1e-12

# Debe existir en tu notebook/script
# METRIC_COLS = ["train_acc", "val_acc", "ood_val_0_acc", "test_acc"]

TARGET_ARCHS = {
    "split",          # profundidad 0
    "split_1",
    "split_2",
    "split_3",
    "split_4",
    "split_resnet_mixer_red_64",   # profundidad 0
    "split_resnet_mixer_red_128",  # profundidad 0
    "split_resnet_mixer_red_256",  # profundidad 0
    *{
        f"split_resnet_mixer_s{s}_rp{rp}_all_cases"
        for s in [1, 2, 3, 4]
        for rp in [64, 128, 256]
    },
}

ROW_ORDER = [
    "AIN",
    "LCIG(AIN)-ReducedRep (64)",
    "LCIG(AIN)-ReducedRep (128)",
    "LCIG(AIN)-ReducedRep (256)",
]

COL_ORDER = ["0", "1", "2", "3", "4"]  # profundidad

def fmt(mean, std, decimals=2):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{decimals}f}"
    return f"{mean:.{decimals}f} ({std:.{decimals}f})"

def parse_arch(arch: str):
    """
    Convierte el nombre del arch en:
      - row_label
      - depth
    """
    arch = str(arch)

    # AIN profundidad 0
    if arch == "split":
        return "AIN", "0"

    # AIN profundidad 1..4
    m = re.fullmatch(r"split_(\d+)", arch)
    if m:
        depth = m.group(1)
        return "AIN", depth

    # ReducedRep profundidad 0
    m = re.fullmatch(r"split_resnet_mixer_red_(\d+)", arch)
    if m:
        rp = int(m.group(1))
        return f"LCIG(AIN)-ReducedRep ({rp})", "0"

    # ReducedRep profundidad 1..4
    m = re.fullmatch(r"split_resnet_mixer_s(\d+)_rp(\d+)_all_cases", arch)
    if m:
        depth = m.group(1)
        rp = int(m.group(2))
        return f"LCIG(AIN)-ReducedRep ({rp})", depth

    return None, None

def plot_ablation_results(dataset, method):
    # =========================
    # CARGA Y FILTRADO
    # =========================
    df = pd.read_pickle(f"{dataset}_{method}.pkl").copy()
    
    if "arch" not in df.columns or "seed" not in df.columns or "test_acc" not in df.columns:
        raise ValueError(f"{dataset}_id.pkl debe contener columnas: arch, seed, test_acc")
    
    df = df[df["arch"].isin(TARGET_ARCHS)].copy()
    
    if df.empty:
        raise ValueError(f"No se encontraron modelos target en {dataset}_{method}.pkl")
    
    # =========================
    # MEJOR CONFIG POR ARCH
    # =========================
    non_metric_cols = [c for c in df.columns if c not in METRIC_COLS]
    config_cols = [c for c in non_metric_cols if c != "seed"]
    
    # 1) promedio por seed dentro de cada configuración
    per_seed = (
        df.groupby(config_cols + ["seed"], dropna=False)["test_acc"]
          .mean()
          .reset_index()
    )
    
    # 2) mean/std entre seeds por configuración
    stats = (
        per_seed.groupby(config_cols, dropna=False)["test_acc"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )
    
    # 3) mejor configuración por arch
    best_idx = stats.groupby("arch")["mean"].idxmax()
    best = stats.loc[best_idx].copy()
    
    # =========================
    # PARSEAR A FILAS/COLUMNAS
    # =========================
    parsed = best["arch"].apply(parse_arch)
    best["row"] = parsed.apply(lambda x: x[0])
    best["depth"] = parsed.apply(lambda x: x[1])
    
    best = best[best["row"].notna() & best["depth"].notna()].copy()
    
    # =========================
    # TABLAS NUMÉRICAS
    # =========================
    mean_df = best.pivot(index="row", columns="depth", values="mean")
    std_df  = best.pivot(index="row", columns="depth", values="std")
    
    mean_df = mean_df.reindex(index=ROW_ORDER, columns=COL_ORDER)
    std_df  = std_df.reindex(index=ROW_ORDER, columns=COL_ORDER)
    
    # máximos por profundidad
    max_by_col = mean_df.max(axis=0, skipna=True)
    is_max = mean_df.sub(max_by_col, axis=1).abs() <= BOLD_TOL
    
    # tabla base como strings
    plain = pd.DataFrame(index=mean_df.index, columns=mean_df.columns, dtype=object)
    for row in plain.index:
        for col in plain.columns:
            plain.loc[row, col] = fmt(mean_df.loc[row, col], std_df.loc[row, col], DECIMALS)
    
    # =========================
    # DISPLAY NOTEBOOK
    # =========================
    display(
        plain.style.apply(
            lambda col: [
                "font-weight: bold" if bool(is_max.loc[idx, col.name]) else ""
                for idx in col.index
            ],
            axis=0,
        )
    )
    
    # =========================
    # MARKDOWN
    # =========================
    md = plain.copy()
    for row in md.index:
        for col in md.columns:
            if bool(is_max.loc[row, col]) and md.loc[row, col] != "":
                md.loc[row, col] = f"**{md.loc[row, col]}**"
    
    print(md.to_markdown())
    
    # =========================
    # LATEX
    # =========================
    tex = plain.copy()
    for row in tex.index:
        for col in tex.columns:
            if bool(is_max.loc[row, col]) and tex.loc[row, col] != "":
                tex.loc[row, col] = r"\textbf{" + tex.loc[row, col] + "}"
    
    tex.index.name = None
    tex.columns.name = None
    
    latex_tabular = tex.to_latex(escape=False, index=True)
    
    latex_small = (
        r"\begin{table}[t]" "\n"
        r"\centering" "\n"
        r"\small" "\n"
        r"\setlength{\tabcolsep}{4pt}" "\n"
        r"\renewcommand{\arraystretch}{1.1}" "\n"
        r"\resizebox{0.8\textwidth}{!}{%" "\n"
        + latex_tabular + "\n"
        r"}" "\n"
        r"\caption{MPI3D results for AIN and LCIG(AIN)-ReducedRep variants across depths, including depth 0 baselines.}" "\n"
        r"\label{tab:mpi3d_depth_results}" "\n"
        r"\end{table}"
    )
    
    print(latex_small)

In [17]:
plot_ablation_results("mpi3d","id")

depth,0,1,2,3,4
row,,,,,
AIN,55.73 (0.31),60.95 (1.57),69.19 (0.75),72.01 (1.38),61.56 (6.19)
LCIG(AIN)-ReducedRep (64),55.57 (0.49),65.00 (6.82),71.01 (1.83),68.96 (6.28),73.38 (1.78)
LCIG(AIN)-ReducedRep (128),56.99 (5.46),63.14 (9.21),67.42 (9.70),72.49 (0.92),68.74 (8.55)
LCIG(AIN)-ReducedRep (256),51.97 (5.59),52.92 (1.07),71.63 (2.85),69.57 (3.58),47.33 (14.45)


| row                        | 0                | 1                | 2                | 3                | 4                |
|:---------------------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|
| AIN                        | 55.73 (0.31)     | 60.95 (1.57)     | 69.19 (0.75)     | 72.01 (1.38)     | 61.56 (6.19)     |
| LCIG(AIN)-ReducedRep (64)  | 55.57 (0.49)     | **65.00 (6.82)** | 71.01 (1.83)     | 68.96 (6.28)     | **73.38 (1.78)** |
| LCIG(AIN)-ReducedRep (128) | **56.99 (5.46)** | 63.14 (9.21)     | 67.42 (9.70)     | **72.49 (0.92)** | 68.74 (8.55)     |
| LCIG(AIN)-ReducedRep (256) | 51.97 (5.59)     | 52.92 (1.07)     | **71.63 (2.85)** | 69.57 (3.58)     | 47.33 (14.45)    |
\begin{table}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.1}
\resizebox{0.8\textwidth}{!}{%
\begin{tabular}{llllll}
\toprule
 & 0 & 1 & 2 & 3 & 4 \\
\midrule
AIN & 55.73 (0.31) & 60.95 (1.57) & 69.19 (0.75) &

In [18]:
plot_ablation_results("mpi3d","oracle")

depth,0,1,2,3,4
row,,,,,
AIN,55.80 (0.34),61.40 (1.07),70.84 (0.32),73.72 (0.71),65.01 (5.77)
LCIG(AIN)-ReducedRep (64),57.30 (2.30),70.01 (0.60),71.99 (0.84),72.41 (0.88),74.44 (1.82)
LCIG(AIN)-ReducedRep (128),58.75 (4.22),65.46 (5.37),73.51 (1.90),73.55 (0.95),69.19 (8.95)
LCIG(AIN)-ReducedRep (256),52.91 (4.32),59.03 (6.85),72.20 (1.90),69.57 (3.58),51.76 (17.42)


| row                        | 0                | 1                | 2                | 3                | 4                |
|:---------------------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|
| AIN                        | 55.80 (0.34)     | 61.40 (1.07)     | 70.84 (0.32)     | **73.72 (0.71)** | 65.01 (5.77)     |
| LCIG(AIN)-ReducedRep (64)  | 57.30 (2.30)     | **70.01 (0.60)** | 71.99 (0.84)     | 72.41 (0.88)     | **74.44 (1.82)** |
| LCIG(AIN)-ReducedRep (128) | **58.75 (4.22)** | 65.46 (5.37)     | **73.51 (1.90)** | 73.55 (0.95)     | 69.19 (8.95)     |
| LCIG(AIN)-ReducedRep (256) | 52.91 (4.32)     | 59.03 (6.85)     | 72.20 (1.90)     | 69.57 (3.58)     | 51.76 (17.42)    |
\begin{table}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.1}
\resizebox{0.8\textwidth}{!}{%
\begin{tabular}{llllll}
\toprule
 & 0 & 1 & 2 & 3 & 4 \\
\midrule
AIN & 55.80 (0.34) & 61.40 (1.07) & 70.84 (0.32) &

## Ablación LCIG

In [26]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
DECIMALS = 2
BOLD_TOL = 1e-12

# Si quieres incluir también el modelo base LCIG(AIN), descomenta la primera línea
ABLATION_ARCHES = [
    #"split_resnet_mixer_all_cases",  # LCIG(AIN) base
    "split_resnet_mixer_all_cases",
    "split_resnet_mixer_all_cases_iid",
    "split_resnet_mixer_no_mixer",
    "split_resnet_mixer_no_mixer_iid",
]

ABLATION_NAME_MAP = {
    arch: MODEL_NAME_MAP[arch]
    for arch in ABLATION_ARCHES
    if arch in MODEL_NAME_MAP
}

# Orden visible de filas
ROW_ORDER = [arch for arch in ABLATION_ARCHES if arch in ABLATION_NAME_MAP]

# ============================================================
# HELPERS
# ============================================================
def safe_score(mean, std):
    if pd.isna(mean):
        return np.nan
    return mean - (0.0 if pd.isna(std) else std)

# ============================================================
# 1) Elegir mejor config por arch y dataset usando mean - std
# ============================================================
records = []

for dataset in DATASETS:
    df = pd.read_pickle(f"{dataset}_id.pkl").copy()

    if "arch" not in df.columns or "seed" not in df.columns or "test_acc" not in df.columns:
        raise ValueError(f"{dataset}_id.pkl debe contener columnas: arch, seed, test_acc")

    # Nos quedamos solo con las arquitecturas de ablación
    df = df[df["arch"].isin(ABLATION_ARCHES)].copy()
    if df.empty:
        continue

    # Todas las columnas no métricas definen la configuración, excepto seed
    non_metric_cols = [c for c in df.columns if c not in METRIC_COLS]
    config_cols = [c for c in non_metric_cols if c != "seed"]

    # Promedio por seed dentro de cada configuración
    per_seed = (
        df.groupby(config_cols + ["seed"], dropna=False)["test_acc"]
        .mean()
        .reset_index()
    )

    # Estadísticos entre seeds para cada configuración
    stats = (
        per_seed.groupby(config_cols, dropna=False)["test_acc"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )

    stats["score"] = stats.apply(lambda r: safe_score(r["mean"], r["std"]), axis=1)

    # Mejor configuración por arch según mean - std
    best_idx = stats.groupby("arch")["score"].idxmax()
    best = stats.loc[best_idx].copy()

    for _, r in best.iterrows():
        records.append({
            "arch": r["arch"],
            "dataset": dataset,
            "mean": r["mean"],
            "std": r["std"],
            "score": r["score"],
        })

long_df = pd.DataFrame(records)

# ============================================================
# 2) Matrices
# ============================================================
mean_df = long_df.pivot(index="arch", columns="dataset", values="mean").reindex(
    index=ROW_ORDER, columns=DATASETS
)
std_df = long_df.pivot(index="arch", columns="dataset", values="std").reindex(
    index=ROW_ORDER, columns=DATASETS
)
score_df = long_df.pivot(index="arch", columns="dataset", values="score").reindex(
    index=ROW_ORDER, columns=DATASETS
)

# Mejor ablación por dataset (usando mean - std)
best_score_by_dataset = score_df.max(axis=0, skipna=True)
is_best = score_df.sub(best_score_by_dataset, axis=1).abs() <= BOLD_TOL

# ============================================================
# 3) Tabla base
# ============================================================
plain = pd.DataFrame(index=ROW_ORDER, columns=DATASETS, dtype=object)

for ds in DATASETS:
    for arch in ROW_ORDER:
        plain.loc[arch, ds] = fmt(
            mean_df.loc[arch, ds],
            std_df.loc[arch, ds],
            decimals=DECIMALS,
        )

# ============================================================
# 4) NOTEBOOK DISPLAY
# ============================================================
plain_disp = plain.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)

is_best_disp = (
    is_best.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)
    .reindex(index=plain_disp.index, columns=plain_disp.columns)
    .fillna(False)
)

display(
    plain_disp.style.apply(
        lambda col: [
            "font-weight: bold" if bool(is_best_disp.loc[idx, col.name]) else ""
            for idx in col.index
        ],
        axis=0,
    )
)

# ============================================================
# 5) MARKDOWN
# ============================================================
md = plain.copy()

for ds in DATASETS:
    for arch in ROW_ORDER:
        if bool(is_best.loc[arch, ds]) and md.loc[arch, ds] != "":
            md.loc[arch, ds] = f"**{md.loc[arch, ds]}**"

md = md.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)
print(md.to_markdown())

# ============================================================
# 6) LATEX
# ============================================================
tex = plain.copy()

for ds in DATASETS:
    for arch in ROW_ORDER:
        if bool(is_best.loc[arch, ds]) and tex.loc[arch, ds] != "":
            tex.loc[arch, ds] = r"\textbf{" + tex.loc[arch, ds] + "}"

tex = tex.rename(index=ABLATION_NAME_MAP, columns=DATASET_NAME_MAP)

dataset_headers = [DATASET_NAME_MAP.get(ds, ds) for ds in DATASETS]
colspec = "l" + "c" * len(DATASETS)

latex_lines = []
latex_lines.append(r"\begin{table}[t]")
latex_lines.append(r"\centering")
latex_lines.append(r"\small")
latex_lines.append(r"\setlength{\tabcolsep}{4pt}")
latex_lines.append(r"\renewcommand{\arraystretch}{1.1}")
latex_lines.append(r"\resizebox{\textwidth}{!}{%")
latex_lines.append(rf"\begin{{tabular}}{{{colspec}}}")
latex_lines.append(r"\toprule")
latex_lines.append("Method & " + " & ".join(dataset_headers) + r" \\")
latex_lines.append(r"\midrule")

for arch in ROW_ORDER:
    row_name = ABLATION_NAME_MAP.get(arch, arch)
    cells = [tex.loc[row_name, DATASET_NAME_MAP.get(ds, ds)] for ds in DATASETS]
    latex_lines.append(row_name + " & " + " & ".join(cells) + r" \\")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"}")
latex_lines.append(r"\caption{Ablation results for LCIG(AIN). For each method, we select the best internal configuration using $\mathrm{mean} - \mathrm{std}$ on the selected model, and report test accuracy. Bold indicates the best ablation per dataset.}")
latex_lines.append(r"\label{tab:lcig_ain_ablation}")
latex_lines.append(r"\end{table}")

latex_small = "\n".join(latex_lines)
print(latex_small)

,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
LCIG(AIN) [ALL],47.37 (1.41),62.92 (2.01),55.84 (1.30),59.93 (4.67),70.31 (2.71),91.91 (6.07)
LCIG(AIN) [ALL] + IID,48.79 (0.27),56.84 (1.20),54.71 (0.08),43.77 (29.34),69.48 (8.26),93.95 (4.46)
LCIG(AIN) - MIXER,48.22 (1.33),60.87 (5.01),60.19 (6.84),56.66 (6.62),62.89 (3.91),93.26 (2.85)
LCIG(AIN) - MIXER + IID,47.92 (1.68),60.53 (3.07),53.82 (2.29),56.03 (3.64),61.71 (5.94),90.49 (4.17)


|                         | C3D              | dSprites         | MPI3D            | CLEVR            | I-RAVEN          | Sh3D             |
|:------------------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|
| LCIG(AIN) [ALL]         | 47.37 (1.41)     | **62.92 (2.01)** | 55.84 (1.30)     | **59.93 (4.67)** | **70.31 (2.71)** | 91.91 (6.07)     |
| LCIG(AIN) [ALL] + IID   | **48.79 (0.27)** | 56.84 (1.20)     | **54.71 (0.08)** | 43.77 (29.34)    | 69.48 (8.26)     | 93.95 (4.46)     |
| LCIG(AIN) - MIXER       | 48.22 (1.33)     | 60.87 (5.01)     | 60.19 (6.84)     | 56.66 (6.62)     | 62.89 (3.91)     | **93.26 (2.85)** |
| LCIG(AIN) - MIXER + IID | 47.92 (1.68)     | 60.53 (3.07)     | 53.82 (2.29)     | 56.03 (3.64)     | 61.71 (5.94)     | 90.49 (4.17)     |
\begin{table}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.1}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lcccccc